# AIMDL API Endpoints

This notebook calls each `aimdl/` REST endpoint directly using the Girder client, showing what parameters each accepts and what the raw response looks like.

| Endpoint | Description |
|---|---|
| `GET /aimdl/count` | Count data files per type across the collection |
| `GET /aimdl/datatype` | List all available data types |
| `GET /aimdl/datafiles` | Query items by data type (pagination, sort, filter, extra fields) |
| `GET /aimdl/partition` | List partitions that have data for a given type |
| `GET /aimdl/partition/details` | Get Dagster partition details for a specific IGSN |

## Setup

Authenticate directly with the Girder client. Reads `GIRDER_API_URL` and `GIRDER_API_KEY` from the `.env` file at the project root.

In [2]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from girder_client import GirderClient

load_dotenv(find_dotenv())

gc = GirderClient(apiUrl=os.environ['GIRDER_API_URL'])
gc.authenticate(apiKey=os.environ[3KRvR25OcbcxWDMa76WnSSkqaDfnH5GK1r4q5g90])
login = gc.get('user/me')['login']
print('Authenticated as:', login)

SyntaxError: invalid syntax (1810485600.py, line 10)

## `GET /aimdl/count`

Returns the number of data files per type in the AIMDL collection. Accepts optional `baseParentId` and `baseParentType` to scope the count to a specific Girder collection.

In [ ]:
counts = gc.get('aimdl/count')
print(json.dumps(counts, indent=2))

## `GET /aimdl/datatype`

Returns an array of all data type strings registered in the AIMDL collection. No parameters required.

In [ ]:
data_types = gc.get('aimdl/datatype')
print(json.dumps(data_types, indent=2))

## `GET /aimdl/datafiles`

The main query endpoint. `dataType` is required; everything else is optional.

| Parameter | Default | Description |
|---|---|---|
| `dataType` | — | **Required.** The data type to filter by |
| `limit` | 50 | Page size |
| `offset` | 0 | Pagination offset |
| `sort` | `lowerName` | Field to sort by |
| `sortdir` | 1 | 1 = ascending, -1 = descending |
| `extraFields` | — | JSON array of dotted field paths to include (e.g. `["meta.prov", "meta.runId"]`) |
| `filters` | — | JSON object of additional server-side filters |
| `baseParentId` / `baseParentType` | — | Scope to a specific Girder collection |

In [ ]:
# Basic call — fetch 3 pdv_alpss_result items
items = gc.get('aimdl/datafiles', parameters={
    'dataType': 'pdv_alpss_result',
    'limit': 3,
})

print(f'Fetched {len(items)} items')
print()
print('First item (default fields):')
print(json.dumps(items[0], indent=2))

### Sorting

Use `sort` and `sortdir` to control result order. `sortdir=-1` gives descending order (most recently created first).

In [ ]:
recent = gc.get('aimdl/datafiles', parameters={
    'dataType': 'pdv_alpss_result',
    'limit': 5,
    'sort': 'created',
    'sortdir': -1,
})

print('5 most recently created pdv_alpss_result items:')
for item in recent:
    print(' ', item['name'], '—', item.get('created', ''))

### Extra fields

`extraFields` is a JSON-encoded array of dotted field paths. Use it to include metadata not returned by default, such as provenance info or run IDs.

In [ ]:
items_with_prov = gc.get('aimdl/datafiles', parameters={
    'dataType': 'pdv_alpss_result',
    'limit': 1,
    'extraFields': '["meta.prov", "meta.runId"]',
})

item = items_with_prov[0]
print('meta with extra fields:')
print(json.dumps(item.get('meta', {}), indent=2))

## `GET /aimdl/partition`

Returns a dict of partition keys (formatted as `IGSN//Date`) mapped to checksums.

In [ ]:
partitions = gc.get('aimdl/partition', parameters={
    'dataType': 'pdv_alpss_result',
})

# The response is a dict keyed by IGSN — use list() to get the keys as a list
print('Response type:', type(partitions).__name__)
igsn_list = list(partitions.keys())
print(f'Found {len(igsn_list)} partitions with pdv_alpss_result data')
print('First 5:', igsn_list[:5])

The optional `since` parameter (ISO 8601 date string) limits results to partitions updated after that date — handy for incremental syncs

In [ ]:
# The `since` parameter filters to items updated after a date (ISO 8601)
recent_partitions = gc.get('aimdl/partition', parameters={
    'dataType': 'pdv_alpss_result',
    'since': '2025-01-01T00:00:00',
})

print(f'IGSNs with pdv_alpss_result updated since 2025-01-01: {len(recent_partitions)}')

### Understanding Partition Keys

Each partition key combines an IGSN (sample identifier) and experiment date, formatted as `IGSN//Date`. For example: `JHACRD00005-S10//2026-03-20`. The value is a checksum of that partition's content, enabling you to:
- Track which samples have data for a given data type
- Detect when data changes (checksums will differ)
- Fetch data incrementally organized by sample and date

In [ ]:
# Inspect partition structure
print('Partition key format: IGSN//Date → Checksum')
for key, checksum in list(partitions.items())[:3]:
    igsn, date = key.split('//')
    print(f'  {igsn} ({date}) → {checksum}')

## `GET /aimdl/partition/details`

Returns Dagster partition details for a specific IGSN. The `key` parameter is an IGSN string from the `/aimdl/partition` response. Use this to inspect what data has been processed for a single sample.

In [ ]:
# Use the first IGSN from the list above
first_igsn = igsn_list[0]
print('Fetching details for IGSN:', first_igsn)

details = gc.get('aimdl/partition/details', parameters={
    'key': first_igsn,
    'dataType': 'pdv_alpss_result',
})

print(json.dumps(details, indent=2))

In [ ]:
summary = (
    files.groupby(["igsn", "experiment_date", "technique"], dropna=False)
         ["file_id"].nunique()
         .unstack("technique", fill_value=0)
         .reindex(columns=["XRD", "PDV"], fill_value=0)
         .rename(columns={"XRD": "xrd_file_count",
                          "PDV": "pdv_file_count"})
         .reset_index()
         .merge(samples[["igsn", "title"]], on="igsn", how="left")
)

In [ ]:
import json
import re
from datetime import datetime

import pandas as pd
from IPython.display import display


def api_pages(endpoint, parameters=None, page_size=100):
    """Retrieve every page using the authenticated client from the tutorial."""
    offset = 0
    while True:
        batch = gc.get(endpoint, parameters={
            **(parameters or {}),
            "limit": page_size,
            "offset": offset,
            "sort": "_id",
            "sortdir": 1,
        })
        if not isinstance(batch, list):
            raise TypeError(f"{endpoint} returned {type(batch).__name__}, not a list")
        if not batch:
            break
        yield from batch
        offset += len(batch)


sample_rows = []
for record in api_pages("deposition"):
    title = " | ".join(
        t.get("title", "")
        for t in (record.get("metadata") or {}).get("titles", [])
    )
    if "Ti" in title or "titanium" in title.lower():
        if record.get("igsn"):
            sample_rows.append({
                "igsn": record["igsn"],
                "title": title,
            })

samples = (
    pd.DataFrame(sample_rows, columns=["igsn", "title"])
      .drop_duplicates("igsn")
      .sort_values("igsn")
      .reset_index(drop=True)
)

print(f"{len(samples)} IGSN records with matching titles")
display(samples)

In [ ]:
if samples.empty:
    raise ValueError("No matching titles found. Review the title-matching rule.")

title_lookup = samples.set_index("igsn")["title"].to_dict()
matched_ids = sorted(title_lookup, key=len, reverse=True)

# Avoid querying a child separately when a matching ancestor covers it.
query_roots = [
    igsn for igsn in matched_ids
    if not any(
        igsn.startswith(other + "-")
        for other in matched_ids
        if other != igsn
    )
]

def title_source(igsn):
    return next(
        (s for s in matched_ids if igsn == s or igsn.startswith(s + "-")),
        None,
    )

def experiment_day(value):
    if not value:
        return "UNKNOWN_DATE"
    try:
        return datetime.fromisoformat(
            str(value).strip().replace("Z", "+00:00")
        ).date().isoformat()
    except ValueError:
        return "INVALID_DATE"


# Collect unique items first; no experimental data bytes are downloaded.
raw_items = {}

for root in sorted(query_roots):
    for data_type, technique in [
        ("xrd_raw", "XRD"),
        ("pdv_trace", "PDV"),
    ]:
        parameters = {
            "dataType": data_type,
            "filters": json.dumps({
                "meta.igsn": {"$regex": "^" + re.escape(root) + r"($|-)"},
            }),
            "extraFields": json.dumps(["meta.experiment_date"]),
        }
        for item in api_pages("aimdl/datafiles", parameters):
            raw_items[item["_id"]] = (technique, item)
    print(f"Queried {root}; {len(raw_items)} unique raw-data items so far")

file_rows = []
empty_items = []

for number, (item_id, (technique, item)) in enumerate(raw_items.items(), 1):
    meta = item.get("meta") or {}
    igsn = meta.get("igsn", "")
    source = title_source(igsn)
    if source is None:
        raise ValueError(f"Unexpected IGSN returned by filter: {igsn!r}")

    item_files = list(api_pages(f"item/{item_id}/files"))
    if not item_files:
        empty_items.append(item_id)

    for file_record in item_files:
        file_rows.append({
            "igsn": igsn,
            "title": title_lookup[source],
            "title_source_igsn": source,
            "experiment_date": experiment_day(meta.get("experiment_date")),
            "experiment_date_original": meta.get("experiment_date"),
            "technique": technique,
            "item_id": item_id,
            "file_id": file_record["_id"],
            "file_name": file_record["name"],
        })

    if number % 100 == 0:
        print(f"Listed files for {number}/{len(raw_items)} items")

files = pd.DataFrame(file_rows, columns=[
    "igsn", "title", "title_source_igsn", "experiment_date",
    "experiment_date_original", "technique", "item_id", "file_id", "file_name",
]).drop_duplicates(["igsn", "technique", "file_id"])

print(f"{len(files)} associated files; {len(empty_items)} items without files")
display(files.head())